# BayesBreak Quickstart Tutorial

This notebook demonstrates the core functionality of **BayesBreak**, a Python package for Bayesian piecewise-constant regression (segmentation) via dynamic programming.

## What is BayesBreak?

BayesBreak implements exact Bayesian inference for piecewise-constant signals, providing:
- **Posterior over segment count** $P(k|\mathbf{y})$
- **Marginal posterior over boundary locations**
- **MAP-like piecewise-constant fits**
- **Bayesian regression curves** that average over uncertainty

The algorithm uses dynamic programming with $O(k_{\max} n^2)$ complexity.

## 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# BayesBreak imports
from bayesbreak import BayesBreakGaussian, BayesBreakPoisson, BayesBreakBernoulli

# Set random seed for reproducibility
rng = np.random.default_rng(42)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)

## 2. Generate Synthetic Piecewise-Constant Data

We'll create a signal with three distinct segments corrupted by Gaussian noise.

In [ ]:
# True segment parameters
true_means = [0.0, 2.5, -1.0]
segment_lengths = [80, 60, 70]
noise_std = 0.4

# Generate piecewise-constant signal
true_signal = np.concatenate([np.full(n, mu) for mu, n in zip(true_means, segment_lengths)])
y = true_signal + noise_std * rng.standard_normal(len(true_signal))

# True boundaries (0-indexed positions where segments end)
true_boundaries = np.cumsum(segment_lengths[:-1]).tolist()

print(f"Total observations: {len(y)}")
print(f"True boundaries: {true_boundaries}")
print(f"True segment means: {true_means}")

In [ ]:
# Visualize the data
fig, ax = plt.subplots()
ax.plot(y, 'o', alpha=0.5, markersize=3, label='Noisy observations')
ax.plot(true_signal, 'k-', linewidth=2, label='True signal')
for b in true_boundaries:
    ax.axvline(b, color='red', linestyle='--', alpha=0.7)
ax.set_xlabel('Index')
ax.set_ylabel('Value')
ax.set_title('Synthetic Piecewise-Constant Data')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Fit BayesBreak Gaussian Model

The `BayesBreakGaussian` class implements the Normal-Normal conjugate model.

In [ ]:
# Create and fit the model
model = BayesBreakGaussian(
    k_max=15,                    # Maximum number of segments to consider
    regression_curve="mix_k",   # Compute Bayesian regression curve
)
model.fit(y)

print(f"Estimated number of segments: {model.get_segment_count()}")
print(f"Estimated boundaries: {model.get_boundaries()}")

## 4. Posterior Over Segment Count

BayesBreak computes the full posterior distribution $P(k|\mathbf{y})$.

In [ ]:
# Get posterior over k
k_values = np.arange(1, model.k_max + 1)
k_posterior = model.C_  # P(k|y) for k=1,...,k_max

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(k_values, k_posterior, color='steelblue', alpha=0.7)
ax.axvline(3, color='red', linestyle='--', label='True k=3')
ax.axvline(model.get_segment_count(), color='green', linestyle='--', label=f'Selected k={model.get_segment_count()}')
ax.set_xlabel('Number of segments (k)')
ax.set_ylabel('P(k | y)')
ax.set_title('Posterior Distribution Over Segment Count')
ax.set_xlim(0.5, 10.5)
ax.legend()
plt.tight_layout()
plt.show()

## 5. Boundary Posterior Probabilities

The marginal posterior probability that each position is a boundary.

In [ ]:
# Get boundary posterior
boundary_posterior = model.boundary_post_

fig, ax = plt.subplots(figsize=(12, 3))
ax.fill_between(range(len(boundary_posterior)), boundary_posterior, alpha=0.6, color='steelblue')
for b in true_boundaries:
    ax.axvline(b, color='red', linestyle='--', alpha=0.8, label='True boundary' if b == true_boundaries[0] else '')
ax.set_xlabel('Index')
ax.set_ylabel('P(boundary | y)')
ax.set_title('Marginal Boundary Posterior Probabilities')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Piecewise-Constant Fit and Bayesian Regression Curve

In [ ]:
# Get predictions
pc_fit = model.predict()              # MAP-like piecewise-constant fit
brc = model.get_regression_curve()    # Bayesian regression curve (averages over k)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(y, 'o', alpha=0.3, markersize=3, label='Observations', color='gray')
ax.plot(true_signal, 'k-', linewidth=2, label='True signal')
ax.plot(pc_fit, 'b-', linewidth=2, label='Piecewise-constant fit')
ax.plot(brc, 'r-', linewidth=2, alpha=0.8, label='Bayesian regression curve')

# Mark estimated boundaries
for b in model.get_boundaries():
    ax.axvline(b, color='blue', linestyle=':', alpha=0.5)

ax.set_xlabel('Index')
ax.set_ylabel('Value')
ax.set_title('BayesBreak Segmentation Results')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Sample Weights

BayesBreak supports per-observation weights for handling heteroscedasticity or missing data.

In [ ]:
# Create weights: downweight noisy observations
weights = np.ones_like(y)
weights[100:130] = 0.1  # Low confidence region

# Fit with weights
model_weighted = BayesBreakGaussian(k_max=15)
model_weighted.fit(y, sample_weight=weights)

print(f"Weighted model - segments: {model_weighted.get_segment_count()}")
print(f"Weighted model - boundaries: {model_weighted.get_boundaries()}")

## 8. Other Likelihood Families

BayesBreak supports multiple conjugate families for different data types.

### 8.1 Poisson (Count Data)

In [ ]:
# Generate Poisson count data
true_rates = [2.0, 8.0, 3.0]
y_poisson = np.concatenate([
    rng.poisson(rate, size=n) 
    for rate, n in zip(true_rates, segment_lengths)
])

# Fit Poisson model
model_poisson = BayesBreakPoisson(k_max=10)
model_poisson.fit(y_poisson)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y_poisson, 'o', alpha=0.5, markersize=3, label='Count data')
ax.plot(model_poisson.predict(), 'r-', linewidth=2, label='Poisson fit')
ax.set_xlabel('Index')
ax.set_ylabel('Count')
ax.set_title(f'Poisson Segmentation (k={model_poisson.get_segment_count()})')
ax.legend()
plt.tight_layout()
plt.show()

### 8.2 Bernoulli (Binary Data)

In [ ]:
# Generate binary data
true_probs = [0.2, 0.8, 0.3]
y_binary = np.concatenate([
    rng.binomial(1, p, size=n) 
    for p, n in zip(true_probs, segment_lengths)
])

# Fit Bernoulli model
model_bernoulli = BayesBreakBernoulli(k_max=10)
model_bernoulli.fit(y_binary)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y_binary, 'o', alpha=0.3, markersize=3, label='Binary data')
ax.plot(model_bernoulli.predict(), 'r-', linewidth=2, label='Bernoulli fit')
ax.set_xlabel('Index')
ax.set_ylabel('Value')
ax.set_title(f'Bernoulli Segmentation (k={model_bernoulli.get_segment_count()})')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Summary

**Key BayesBreak features:**

| Method | Description |
|--------|-------------|
| `fit(y)` | Fit the model to data |
| `predict()` | MAP-like piecewise-constant reconstruction |
| `get_segment_count()` | Estimated number of segments |
| `get_boundaries()` | Estimated boundary positions |
| `get_regression_curve()` | Bayesian regression curve |
| `boundary_post_` | Marginal boundary posteriors |
| `C_` | Posterior P(k|y) |

**Available families:**
- `BayesBreakGaussian` - continuous data
- `BayesBreakPoisson` - count data
- `BayesBreakBernoulli` - binary data
- `BayesBreakBinomial` - binomial trials
- `BayesBreakBeta` - fractional/proportion data